In [ ]:
# Check GPU, make sure it use A100
!nvidia-smi

Tue May 12 03:56:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   30C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
import os
print(os.getcwd())

/content


In [ ]:
!pip install transformers accelerate bitsandbytes omegaconf pandas

In [ ]:
# Indonesian
from omegaconf import OmegaConf

cfg = OmegaConf.create({
    "model_name": "aya-expanse-8b",
    "model_path": "CohereForAI/aya-expanse-8b",  # HF model name
    "data_path": "/content/gdrive/MyDrive/mental_health/dataset_ind.jsonl",
    "output_dir": "/content/gdrive/MyDrive/mental_health/outputs",
    "cache_dir": "/content/gdrive/MyDrive/hf_cache",
    "culture": "Indonesia",
    "strategy": "guided_annotation",  #change the strategy to redditor, profile, guided_without_profile, annotation_without_profile based on prompting_strategy.py
    "generate_cfg": {
        "temperature": 0.7,
        "max_new_tokens": 1000
    }
})

In [ ]:
import sys
sys.path.append('/content/gdrive/MyDrive/mental_health')

import prompting_strategies

In [ ]:
import json
import os
import pandas as pd
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

# Make sure you had prompting_strategies.py
from prompting_strategies import NAME2STRATEGY, NAME2CULTURE, USER_PROMPT, annotations_prompt

def load_dataset(cfg):
    path = cfg.data_path.format(culture=cfg.culture)
    dataset = pd.read_json(cfg.data_path, lines=True)
    print(f"Columns of the data {dataset.columns}")
    print(f"Size of the data {len(dataset)}")
    return dataset.to_dict(orient="records")

class ResponseGenerator:
    def __init__(self, cfg):
        self.cfg = cfg

    def load_model(self):
        quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.cfg.model_path,
            quantization_config=quantization_config,
            device_map="auto",
            cache_dir=self.cfg.cache_dir,
            local_files_only=False
        )

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.cfg.model_path,
            cache_dir="/content/hf_cache",
            local_files_only=False
        )

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def _generate(self, prompt):
        self.model.eval()
        prompt = self.tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
        if self.tokenizer.bos_token:
            prompt = prompt.replace(self.tokenizer.bos_token, "")
            prompt_tokenized = self.tokenizer.encode(prompt, return_tensors="pt").to(self.model.device)

        prompt_tokenized = self.tokenizer.encode(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            output_tokenized = self.model.generate(prompt_tokenized, pad_token_id=self.tokenizer.eos_token_id,
                                                   **self.cfg.generate_cfg)

        output = self.tokenizer.decode(output_tokenized[0], skip_special_tokens=True)
        output_o = output.replace(str(self.tokenizer.bos_token), "").replace(str(self.tokenizer.eos_token), "").strip()
        model_prompt_o = prompt.replace(str(self.tokenizer.bos_token), "").replace(str(self.tokenizer.eos_token),
                                                                                   "").strip()
        response = output_o.replace(model_prompt_o, "", 1)
        return response.split("**Response**:")[-1].lstrip('assistant_').lstrip("assistant").strip()

    def _get_user_prompts(self, datapoint):
        if "annotation" in self.cfg.strategy:
            user_prompt = annotations_prompt(datapoint)
        else:
            user_prompt = USER_PROMPT.format(post=datapoint["post"]["text"])
        return user_prompt

    def _get_prompts(self, datapoint):
        culture = NAME2CULTURE.get(self.cfg.culture.lower(), self.cfg.culture)
        strategy_template = NAME2STRATEGY.get(self.cfg.strategy)
        sys_prompt = strategy_template.format(culture=culture)
        user_prompt = self._get_user_prompts(datapoint)

        return [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_prompt},
        ]

    def generate_response(self, dataset):
        output_dir = os.path.join(self.cfg.output_dir, self.cfg.model_name, self.cfg.strategy)

        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
            print(f"Directory '{output_dir}' created.")
        else:
            print(f"Directory '{output_dir}' already exists.")

        output_file = open(os.path.join(self.cfg.output_dir, self.cfg.model_name, self.cfg.strategy, f"{self.cfg.culture}_response.json"), 'a')

        for idx, datapoint in enumerate(dataset):
            if idx % 5 == 0:
                print(f"Current progress: {idx}")
            prompt = self._get_prompts(datapoint)
            results = self._generate(prompt)
            out_data = {
                "post_id": datapoint["post_id"],
                "post": datapoint["post"]["text"],
                "response": results
            }
            torch.cuda.empty_cache()
            output_file.write(json.dumps(out_data) + "\n")


In [ ]:
dataset = load_dataset(cfg)

Columns of the data Index(['culture', 'post_id', 'post', 'response'], dtype='object')
Size of the data 20


In [ ]:
from huggingface_hub import login
login()

In [ ]:
generator = ResponseGenerator(cfg)
generator.load_model()

print("Model loaded")

generator.generate_response(dataset)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Model loaded
Directory '/content/gdrive/MyDrive/mental_health/outputs/aya-expanse-8b/annotation_without_profile' already exists.
Current progress: 0
Current progress: 5
Current progress: 10
Current progress: 15
